# Propuesta de laboratorio flow matching MML 2026

## Fundamentos teóricos

1. **La ecuación de flujo**:
   Como en los CNFs:

   $$\frac{dx_t}{dt} = v_\theta(x_t, t)$$

   Donde:
   - $x_t$ es nuestra imagen mientras se va transformando
   - $t$ es el tiempo (entre 0 y 1); en $t=0$, $p(x_0)$ es $N(0,1)$; en $t=1$, $p(x_1)$ es la distribución objetivo que queremos muestrear.
   - $v_\theta$ es un campo de velocidad que vamos a parametrizar con una red neuronal, y que determinará la dirección en la que hay que moverse para muestrear de la distribución objetivo.

2. **La trayectoria** (interpolación lineal entre ruido y dato)

   Elegimos un interpolante, $x_t$, que cumple:

   $$x_t = (1-t)x_0 + tx_1$$

   Donde:
   - $x_0$ es ruido aleatorio (nuestro punto de partida)
   - $x_1$ es una imagen real de galaxia (nuestro objetivo)

3. **Función de pérdida**:
   Entrenamos nuestro modelo para minimizar:

   $$L = \mathbb{E}_{t,x_0,x_1} \left[ \left\| v_\theta(x_t, t) - u(x_0,x_1,t) \right\|^2 \right]$$

   Donde, dado el interpolante lineal, el campo de velocidad verdadero es $u(x_0, x_1,t) = x_1 - x_0$. Esto le enseña al modelo a predecir la dirección correcta de cambio en cada paso.

## 0. Importación de librerias 

Vamos a importar todas las librerías utilizadas en el laboratorio

In [ ]:
%pip install torch torchvision 
%pip install matplotlib
%pip install numpy
%pip install tqdm

In [ ]:
import torch
import torchdiffeq
import torch.utils.data as data

import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import torch.optim as optim
from IPython.display import clear_output

## 1. Creación de datos

Antes de cualquier cosa tenemos que definir un conjunto de datos que vamos a utilizar para el entrenamiento y prueba de la red neuronal. En primera instancia vamos a generar datos de la forma de Two Moons. Estos datos se utilizan para diferentes entrenamientos como clasificación no supervisada y supervisada. Pero en este caso también son lo suficientemente llamativos para permitir una generación de imágenes de la forma 'Two Moons'.

La siguiente función genera n sets de puntos (x,y) que toman la forma de las 'Two moons'. También devuelve la información de clasificación de los puntos; sin embargo, en este caso no vamos a necesitar esa información ya que es para nosotros más valiosa la forma de los puntos en general.

In [ ]:
def two_moons(n: int, sigma: float = 5e-2):
    theta = 2 * torch.pi * torch.rand(n)
    label = (theta > torch.pi).float()

    x = torch.stack(
        (
            torch.cos(theta) + label - 1 / 2,
            torch.sin(theta) + label / 2 - 1 / 4,
        ),
        axis=-1,
    )

    return torch.normal(x, sigma), label


samples, _ = two_moons(16384)

Ahora veamos la forma de los datos

In [ ]:
samples.shape

Y veamos cómo se distribuyen los diferentes puntos de todos los puntos generados

In [ ]:
plt.figure(figsize=(4.8, 4.8))
plt.hist2d(*samples.T, bins=64, range=((-2, 2), (-2, 2)))
plt.show()

## 2. Definición de interpolante y red neuronal

Antes de que definamos una red neuronal para el problema debemos entender cuál es el propósito que queremos que cumpla nuestra red. Por lo tanto, vamos a modelar el movimiento ideal de los datos desde el ruido hasta nuestra distribución final (Two Moons).

In [ ]:
def interpolant(x0, x1, t):
    """

    Implementa la interpolación lineal entre el ruido (x0) y el dato (x1).

    En t=0: deberíamos obtener ruido puro (x0)
    En t=1: deberíamos obtener el dato real (x1)
    En t=0.5: deberíamos obtener una mezcla 50/50

    Args:
        x0: Punto de partida (ruido), shape: (batch_size, ...)
        x1: Punto final (dato), shape: (batch_size, ...)
        t: Parámetro de tiempo, shape: (batch_size,) o escalar

    Returns:
        x_t: Muestras interpoladas en el tiempo t
    """

    # TODO: Implementa la interpolación lineal entre x0 y x1 usando el parámetro t
    pass

fig, axs = plt.subplots(1, 10, figsize=(20, 2))

x1, _ = two_moons(500,)
x1 = torch.tensor(x1).float()
x0  = torch.randn_like(x1)
t = torch.tensor(np.linspace(0., 1., 10))
for i, ax in enumerate(axs):
    xt = interpolant(x0,x1,t[i])
    ax.scatter(xt[:,0], xt[:,1],s=1, color='k')
    ax.axis('off')
    ax.set_title(f't={t[i]:.2f}')

Como se ve en esta imagen los puntos inician distribuidos uniformemente para, poco a poco y a medida que pasa el tiempo, desplazarse a su destino final en una trayectoria lineal.

**Pregunta:** Si podemos generar este movimiento sin entrenar nada, ¿cuál es la razón de necesitar entrenar una red neuronal?

Ahora entendida la necesidad de la red neuronal vamos a modelarla. En la siguiente clase se va a almacenar nuestra red y se necesita definir dos funciones importantes:

1. La función de inicialización: Con la ayuda de las librerías nn de pytorch defina una red neuronal de las siguientes características

    - Una capa de entrada
    - Tres capas ocultas
    - Una capa de salida
    - Funciones de activación SiLU

2. La función forward: Esta función recibirá un punto en el espacio $x_t$ y un tiempo $t$ y los pasará por la red.

**Pregunta:** ¿Cuáles son las dimensiones de las diferentes capas de la red? (Pista: ¿Cuántas neuronas hay en cada capa?)

In [ ]:
import torch.nn as nn

class VelocityNetwork(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        # TODO: Crea una red neuronal que reciba como entrada:
        # - Posición 2D (x_t)
        # - Tiempo 1D (t)
        #
        # Salida: vector de velocidad 2D
        #
        # Guarda esto en self.net como un nn.Sequential

        pass

    def forward(self, xt, t):
        # TODO: Combina xt y t en un único tensor de entrada y pásalo por la
        # red que predice la velocidad
        
        pass

Ahora vamos a crear el modelo e ingresarlo dentro del dispositivo de pytorch. 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

velocity_nn = VelocityNetwork(
    hidden_dim=128,
).to(device)

t=torch.tensor(500*[0.5],dtype=torch.float32, device=device)

#Revisar que la salida de la red neuronal tenga la forma esperada
assert velocity_nn(
    xt=x1.to(device),
    t=t,
).shape == (500, 2), "Esperaba una forma de (500, 2) pero tenía una forma diferente"

## 3. Definir el modelo Flow Matching

La siguiente clase se va a utilizar como definición del modelo de flow matching. En ella van a encontrar diferentes funciones para completar. La más importante que hay que tener en cuenta es la función de pérdida.

Para calcular la función de pérdida:

1. **Muestrea un par de entrenamiento aleatorio**: Toma ruido $x_0$ y dato real $x_1$
2. **Elige un tiempo aleatorio**: Escoge $t$ uniformemente entre 0 y 1 para estimar por Monte Carlo el valor esperado sobre el tiempo.
3. **Obtén el dato interpolado**: $x_t = (1-t)x_0 + tx_1$
4. **Evalúa la red de predicción de velocidad**: "¿Qué velocidad predices en $(x_t, t)$?"
5. **Compara con la verdad, conociendo $x_1$**
6. **Actualiza los pesos**: Usa backpropagation para mejorar las predicciones


In [ ]:
class FlowMatching(nn.Module):
    def __init__(self, velocity_network, device='cuda'):
        super().__init__()
        self.velocity_net = velocity_network
        self.device = device

    def sample_time(self, batch_size):
        """Tomar un tiempo aleatorio t entre 0 y 1 para cada muestra en el batch"""
        return torch.rand(batch_size, device=self.device)

    def interpolate(self, x0, x1, t):
        """
        Interpolación lineal entre el ruido (x0) y el dato (x1).
        En t=0: retorna x0 (ruido)
        En t=1: retorna x1 (dato)
        """

        # TODO: Llenar el interpolante con las dimensiones correctas
        pass

    def compute_target_velocity(self, x0, x1, t):
        """
        Calcula el campo de velocidad objetivo que la red neuronal debe aprender.

        Piensa en esto: si tenemos x_t = (1-t)*x0 + t*x1,
        ¿cuál es dx_t/dt (la derivada con respecto al tiempo)?
        """
        # TODO: completa tu implementación aquí
        pass

    def compute_loss(self, x1,):
        """
        Calcula la pérdida de flow matching.

        Args:
            x1: muestras de datos

        Returns:
            Valor de la pérdida
        """

        # TODO: Calcular la pérdida de flow matching usando las funciones definidas anteriormente
        pass


    def sample(self, target_shape=(2,), num_samples=10, num_steps=200):

      """
        Genera muestras integrando el campo de velocidad aprendido.

        Args:
            num_samples: Número de muestras a generar
            num_steps: Número de pasos de integración

        Returns:
            Muestras generadas
        """


      self.velocity_net.eval()

      x0 = torch.randn(num_samples, *target_shape, device=self.device)
      t = torch.tensor([0.0, 1.0], device=self.device)

      def f(t, x):
          t_batch = t.repeat(num_samples)
          return self.velocity_net(x, t_batch)

      return torchdiffeq.odeint(f, x0, t, method='dopri5')[-1]


Ahora vamos a instanciar el modelo

In [ ]:
fm = FlowMatching(
    velocity_network=velocity_nn,
    device=device,
)

Primero, verifiquemos que podemos llamar a compute_loss. Si nuestros datos están estandarizados correctamente, deberíamos esperar que una red de velocidad inicializada aleatoriamente retorne algo del orden de 1. Siempre presta atención a los valores iniciales de la función de pérdida, ya que valores enormes o diminutos usualmente indican bugs.


In [ ]:
fm.compute_loss(
    x1=x1.to(device),
)

Ahora revisemos el solucionador de ODE para generar datos.

In [ ]:
samples = fm.sample(
   num_samples=500,

).detach().cpu().numpy()

assert samples.shape == (500, 2), "Esperaba una forma de (500, 2) pero tenía una forma diferente"

In [ ]:
plt.scatter(
    samples[:,0],
    samples[:,1],
    c='k',
    s=1,
)

**Pregunta:** Porque los datos tienen una distribución aleatoria

Ahora vamos a entrenar la red

In [ ]:
optimizer = torch.optim.AdamW(fm.parameters(), lr=1.e-4, weight_decay=1e-4)
total_steps = 40_000
generate_every = 100
batch_size = 128
losses = []

# Puntos del entrenamiento en los que queremos guardar una imagen de las muestras generadas
checkpoint_steps = {0, total_steps // 4, total_steps // 2, total_steps}
snapshots = {}

fm.train()

# Muestra de la red antes de entrenar (step=0)
snapshots[0] = fm.sample(num_samples=500).detach().cpu().numpy()
fm.velocity_net.train()

step = 0
with tqdm(total=total_steps, desc="Training") as pbar:
    while step < total_steps:
        x, label = two_moons(batch_size)
        x = x.to(device)  # Move data to device
        loss = fm.compute_loss(x).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fm.parameters(), max_norm=1.0)


        optimizer.step()
        optimizer.zero_grad()


        pbar.set_postfix({
                'Loss': f'{loss:.4f}',
            })
        pbar.update(1)
        step += 1
        losses.append(loss.detach())

        if step in checkpoint_steps:
            snapshots[step] = fm.sample(num_samples=500).detach().cpu().numpy()
            fm.velocity_net.train()

        if step % generate_every == 0:
            print(f"\nGenerating samples at step {step}...")

            samples = fm.sample(
                num_samples=500,
            ).detach().cpu().numpy()
            clear_output(wait=True)
            plt.scatter(
                x[:,0].cpu().numpy(),
                x[:,1].cpu().numpy(),
                c='indianred',
                s=1,
                alpha=0.8,
                label='Original'
            )
            plt.scatter(
                samples[:,0],
                samples[:,1],
                c='k',
                s=1,
                label='Generated'
            )
            plt.legend()
            plt.title(f'Generated Samples at step {step}')
            plt.axis('off')

            plt.show()
            fm.velocity_net.train()

    losses = torch.stack(losses)

# Comparación final del progreso del entrenamiento: 0%, 25%, 50% y 100%
fig, axs = plt.subplots(1, len(checkpoint_steps), figsize=(4 * len(checkpoint_steps), 4))
for ax, step_i in zip(axs, sorted(checkpoint_steps)):
    pct = int(100 * step_i / total_steps)
    ax.scatter(snapshots[step_i][:, 0], snapshots[step_i][:, 1], c='k', s=1)
    ax.set_title(f'{pct}% del entrenamiento\n(step {step_i})')
    ax.axis('off')
plt.tight_layout()
#Si desea guardar la imagen en su computadora, descomente la siguiente línea y cambie el nombre del archivo según sea necesario
#plt.savefig('progreso_entrenamiento_two_moons.png', dpi=150, bbox_inches='tight')
plt.show()

Ahora generemos unas muestras con la red entrenada

In [ ]:
samples = fm.sample(
    num_samples=5_000,
).detach().cpu().numpy()


plt.figure(figsize=(4.8, 4.8))
plt.hist2d(*samples.T, bins=128, range=((-2, 2), (-2, 2)))
plt.show()

**Pregunta:** Que puede observar de los datos generados por el modelo

Observemos la pérdida con respecto a las épocas

In [ ]:
plt.plot(losses.cpu())

## 4. Preguntas finales

1. Todo lo implementado aquí funciona sobre puntos 2D. ¿Qué tendría que cambiar en la arquitectura de la red de velocidad (no en la lógica de FlowMatching) para que este mismo método funcione sobre imágenes de galaxias de 144x144 píxeles?

2. ¿Qué esperarías que pasara con la calidad de las muestras generadas si aumentas el sigma del ruido en two_moons (la dispersión de los datos reales)? 